# 🍺 Atividade Prática: Medalhão (Bronze → Prata → Ouro) com Spark e Delta Lake

**Disciplina:** Arquitetura de Coleta e Armazenamento de Dados (Object Storage + Spark)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wuldson-franco/arqu_coleta_dados_pos_unipe/blob/main/Atividade_Pratica_Spark/atividade_medalhao_breweries.ipynb) -> *CLIQUE AQUI PARA ABRIR O ARQUIVO NO SEU GOOGLE COLAB*

| | |
|---|---|
| **Nome do aluno** | Natan Nascimento Ayres |
| **Data** | 24/10/2026 |

---

## Como fazer esta atividade

1. **Não edite este arquivo diretamente.** Vá em **Arquivo → Salvar uma cópia no Drive** e trabalhe na sua cópia.
2. Rode as células **em ordem**, de cima para baixo. Se der erro estranho, use **Ambiente de execução → Reiniciar sessão** e rode tudo de novo.
3. As perguntas marcadas com ✏️ são **abertas**: responda com as suas palavras, na célula de texto logo abaixo da pergunta.
4. Ao terminar, rode a célula final de **autoverificação** e envie o link da sua cópia diretamente do seu repositorio no Github (com permissão de leitura).

## O que vamos construir

```
 arquivo .parquet  ──►  BRONZE (Delta)  ──►  PRATA (Delta)  ──►  OURO (Delta)
   (dado cru)           dado como chegou     limpo e corrigido    agregado p/ o negócio
```

**Pergunta de negócio:** _"Quais tipos de cervejaria existem por cidade, província e país?"_

> 💡 **Sobre o uso de IA:** você pode usar IA como apoio (Claude, ChatGPT, Gemini, Copilot…). Mas **você** responde pelo resultado. Duas perguntas no final pedem que você reflita sobre esse uso, então guarde o que perguntou e o que você precisou corrigir.

## 0. Preparando o ambiente

Vamos instalar o Java, o PySpark e o Delta Lake. As versões precisam combinar entre si (PySpark 3.5.1 ↔ Delta 3.1.0). **Esta célula demora 1 a 2 minutos.**

In [ ]:
%%bash

# Instalar Java
apt-get update --fix-missing
apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Instalar PySpark e Delta Lake (versões compatíveis para evitar conflito de dependências)
pip uninstall -y delta-spark
pip install -q pyspark==3.5.1
pip install -q delta-spark==3.1.0

In [ ]:
import os, logging, requests

# Apontamos o Spark para o Java que acabamos de instalar
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName("Atividade_Medalhao_Breweries")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.repl.eagerEval.enabled", True)
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark", spark.version, "pronto ✅")

# Nosso "data lake" de brincadeira: pastas no disco da VM do Colab
base_path = "/content/datalake"
for camada in ["bronze", "silver", "gold"]:
    os.makedirs(os.path.join(base_path, camada), exist_ok=True)
print("Pastas criadas em", base_path)

---
## 1. 🥉 Camada Bronze

Vamos baixar o arquivo cru e guardá-lo na Bronze **sem alterar nada**, só convertendo para o formato Delta.

In [ ]:
parquet_url  = "https://github.com/wuldson-franco/breweries_case/raw/main/analises/bs_bronze.parquet"
parquet_path = os.path.join(base_path, "bronze", "bs_bronze.parquet")
delta_bronze = os.path.join(base_path, "bronze", "bs_bronze_delta")

# Baixar o arquivo Parquet (só se ainda não existir)
if not os.path.exists(parquet_path):
    response = requests.get(parquet_url)
    response.raise_for_status()
    with open(parquet_path, "wb") as f:
        f.write(response.content)

df = spark.read.parquet(parquet_path)

# mode("overwrite") permite rodar a célula mais de uma vez sem erro
df.write.format("delta").mode("overwrite").save(delta_bronze)

print("Linhas na Bronze:", spark.read.format("delta").load(delta_bronze).count())

Agora vamos **olhar por dentro** do que foi gravado. Em um bucket de object storage, cada arquivo seria um _objeto_ e o caminho completo seria a sua _chave_.

In [ ]:
# Lista tudo o que existe dentro da Bronze Delta (caminho relativo e tamanho)
for raiz, _, arquivos in os.walk(delta_bronze):
    for nome in sorted(arquivos):
        caminho = os.path.join(raiz, nome)
        print(f"{os.path.relpath(caminho, base_path):<85} {os.path.getsize(caminho):>10,} bytes")

### ✏️ Q1. Olhando por dentro da Bronze
Observe a listagem acima. **(a)** Que tipos de arquivo você vê e para que serve cada um? **(b)** Se essa pasta estivesse em um bucket S3/MinIO, o que seriam a _chave_ e o _objeto_ de um desses arquivos?

**✏️ Sua resposta:**

**(a)** Dentro de `bs_bronze_delta` aparecem dois tipos de arquivo:
- `part-00000-....snappy.parquet`: são os **dados** de fato, em formato colunar Parquet comprimido com Snappy. É onde ficam as 50 linhas.
- `_delta_log/00000000000000000000.json`: é o **log de transações** do Delta. Cada commit (versão) gera um JSON numerado dizendo quais arquivos Parquet foram adicionados/removidos, o schema e metadados. É ele que transforma "uma pasta com Parquets" em uma tabela.
- Os arquivos `.crc` (ex.: `.part-...parquet.crc`, `00000...crc`) são **checksums** usados para validar a integridade dos arquivos.

**(b)** Num bucket S3/MinIO, a *chave* seria o caminho completo como texto, por exemplo `bronze/bs_bronze_delta/_delta_log/00000000000000000000.json` (dentro do bucket `datalake`), e o *objeto* seria o conteúdo desse arquivo (os bytes do JSON) junto com seus metadados (tamanho, content-type, data de modificação, ETag). Não existe a pasta `_delta_log` como entidade: ela é só parte do nome da chave.

### ✏️ Q2. Por que Bronze e por que Delta?
**(a)** Por que a Bronze deve guardar o dado **como ele chegou**, sem corrigir nada? **(b)** Qual problema o Delta resolve que um monte de arquivos Parquet soltos no storage não resolve?

**✏️ Sua resposta:**

**(a)** A Bronze é a nossa "cópia de segurança" fiel da origem. Se eu corrijo algo já na entrada e a correção estiver errada (como um endereço mal pesquisado), perco o dado original e não consigo voltar atrás. Guardando como chegou eu consigo **reprocessar** a Prata quando a regra mudar, **auditar** o que a fonte realmente mandou e separar problemas da fonte de problemas do meu pipeline.

**(b)** Parquet solto é só arquivo: não tem transação. Se um job falha no meio da escrita, quem lê pode ver metade dos arquivos; dois jobs escrevendo ao mesmo tempo se atropelam; não dá pra fazer `UPDATE`/`DELETE`/`MERGE` sem reescrever tudo na mão; e não há histórico. O Delta adiciona o `_delta_log`, que dá **transações ACID** (ou o commit inteiro aparece, ou nada), **validação/evolução de schema**, operações de **MERGE/UPDATE/DELETE**, **histórico de versões (time travel)** e estatísticas de arquivos que ajudam o Spark a pular dados na leitura.

---
## 2. 🥈 Camada Prata

Aqui o dado é inspecionado, limpo e corrigido. Resolva os passos na ordem.

**2.1** Exiba o _schema_ das colunas com `printSchema`.

In [ ]:
df_bronze = spark.read.format("delta").load(delta_bronze)
df_bronze.printSchema()

**2.2** Faça um `select` do país para verificar **quais países constam na nossa base**. Mostre também **quantas cervejarias** há em cada um.

In [ ]:
# Países presentes na base
df_bronze.select("country").distinct().show(truncate=False)

# Quantidade de cervejarias por país
(df_bronze.groupBy("country")
    .agg(F.count("*").alias("qtd_cervejarias"))
    .orderBy(F.desc("qtd_cervejarias"))
    .show(truncate=False))

**2.3** Descubra **quais cervejarias estão com o endereço faltante**.

> Dica: "faltante" pode ser `null` **e também** texto vazio (`""`) ou só espaços. Confira os dois casos. Qual é a coluna de endereço? Veja no `printSchema`.

In [ ]:
# Endereço faltante = null OU texto vazio / só espaços
cond_faltante = F.col("address_1").isNull() | (F.trim(F.col("address_1")) == "")

print("Nulos:", df_bronze.filter(F.col("address_1").isNull()).count())
print("Vazios/espaços:", df_bronze.filter(F.col("address_1").isNotNull() & (F.trim(F.col("address_1")) == "")).count())

df_bronze.filter(cond_faltante).select(
    "id", "name", "brewery_type", "address_1", "street", "city", "state_province", "postal_code", "country"
).show(truncate=False)

**2.4** Pesquise na internet **qual é o endereço real** dessas cervejarias e inclua-os nos campos faltantes usando `when` / `otherwise`.

Crie também uma coluna `address_fonte` dizendo **de onde veio o dado** (por exemplo: `"original"` ou `"pesquisa manual"`). Guarde esta ideia: ela será cobrada na pergunta Q3.

> Estrutura do `when` (os valores abaixo são só um **exemplo**, troque pelos seus):
> ```python
> df_silver = df.withColumn(
>     "address_1",
>     F.when(F.col("id") == "ID_DA_CERVEJARIA", "Endereço encontrado").otherwise(F.col("address_1"))
> )
> ```

In [ ]:
# Endereços pesquisados manualmente (fontes nos comentários)
enderecos = {
    # 14 Lakes Brewery – Crosslake/MN (Brewbound)
    "b7b68d22-5045-4501-b9bf-ec94946eaffc": "36846 County Road 66",
    # 1850 Brewing Company – Mariposa/CA (Yosemite Mariposa County Tourism)
    "fe6b9893-b93e-43d5-a9f6-3e0c89a3f13c": "5114 State Highway 140",
    # 12 West Brewing – Production Facility, Mesa/AZ (ABC15): só localização aproximada publicada
    "0faa0fb2-fffa-416d-9eab-46f67477c8ef": "Longbow Pkwy near Recker Rd",
}

cond_faltante = F.col("address_1").isNull() | (F.trim(F.col("address_1")) == "")

# 1) Marca a origem ANTES de alterar o endereço
df_silver = df_bronze.withColumn(
    "address_fonte",
    F.when(cond_faltante & (F.col("id") == "0faa0fb2-fffa-416d-9eab-46f67477c8ef"), "pesquisa manual (aproximado)")
     .when(cond_faltante & F.col("id").isin(list(enderecos.keys())), "pesquisa manual")
     .when(cond_faltante, "faltante")
     .otherwise("original")
)

# 2) Preenche o endereço com when/otherwise, só onde estava faltando
expr = F.col("address_1")
for _id, end in enderecos.items():
    expr = F.when(cond_faltante & (F.col("id") == _id), F.lit(end)).otherwise(expr)
df_silver = df_silver.withColumn("address_1", expr)

# Mantém 'street' coerente com address_1
df_silver = df_silver.withColumn("street", F.coalesce(F.col("street"), F.col("address_1")))

df_silver.filter(F.col("address_fonte") != "original") \
    .select("id", "name", "address_1", "street", "city", "address_fonte").show(truncate=False)

**2.5** Verificações de qualidade. Responda com código:
- Existem `id` **duplicados**?
- Existem nulos em `brewery_type`, `city` ou `country`?
- Depois da sua correção, ainda sobra algum endereço faltante?

In [ ]:
# ids duplicados?
total = df_silver.count()
distintos = df_silver.select("id").distinct().count()
print(f"Total: {total} | ids distintos: {distintos} | duplicados: {total - distintos}")
df_silver.groupBy("id").count().filter("count > 1").show()

# Nulos (ou vazios) nas colunas-chave da agregação
df_silver.select([
    F.sum((F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")).alias(f"nulos_{c}")
    for c in ["brewery_type", "city", "state_province", "country"]
]).show()

# Ainda sobra endereço faltante?
restantes = df_silver.filter(F.col("address_1").isNull() | (F.trim(F.col("address_1")) == "")).count()
print("Endereços faltantes após correção:", restantes)
assert restantes == 0 and total == distintos, "Falha na validação da Prata"
print("Validações OK ✅")

**2.6** Salve a Prata em **Delta**, em `silver/breweries_silver`, e depois leia de volta para conferir a contagem.

In [ ]:
silver_path = os.path.join(base_path, "silver", "breweries_silver")

# Gravar a Prata em Delta
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)

# Ler de volta e conferir
df_silver_lida = spark.read.format("delta").load(silver_path)
print("Linhas na Prata:", df_silver_lida.count())
df_silver_lida.groupBy("address_fonte").count().show()

### ✏️ Q3. Qualidade e governança
**(a)** Você corrigiu os endereços na mão, com `when/otherwise`. Isso funcionaria para 5 milhões de registros com problemas? Que alternativa você proporia? **(b)** Para que serve a coluna `address_fonte` que você criou? Relacione com os conceitos de **linhagem** e **confiança no dado**.

**✏️ Sua resposta:**

**(a)** Não. Para 3 registros o `when/otherwise` funciona, mas para 5 milhões seria inviável manter (código gigante, impossível pesquisar à mão, e um `when` encadeado enorme fica lento e frágil). Eu proporia: manter uma **tabela de referência/correções** (id → endereço, fonte, data) e aplicá-la com um **join ou `MERGE` do Delta**; enriquecer automaticamente via **API de geocodificação** (reverse geocoding a partir de latitude/longitude, que esta base tem para a maioria das linhas) ou cruzamento com outra base oficial; e o que não for resolvido vai para uma **tabela de quarentena** com flag, em vez de inventar valor. Regras de qualidade (expectations) automatizadas no pipeline medem a taxa de faltantes a cada carga.

**(b)** A `address_fonte` registra **de onde veio cada endereço**: original da API, pesquisa manual ou aproximado. Isso é **linhagem no nível do registro**: quem consome a Prata/Ouro sabe que aquele valor não veio da fonte e pode decidir se confia nele (ex.: não usar endereço "aproximado" para entregas). Também permite auditar e desfazer a correção, e medir quanto da base depende de intervenção manual. Sem essa coluna, o dado corrigido fica indistinguível do original e a confiança na tabela inteira cai.

---
## 3. 🥇 Camada Ouro

Vamos responder à pergunta de negócio: **"Quais tipos de cervejaria por cidade, província e país?"**

Abaixo está a função de agregação, escrita em **pandas**. Ela não roda direto em um DataFrame do Spark. Sua tarefa é **convertê-la para Spark**.

In [ ]:
# ATENÇÃO: esta função é pandas. Não rode com o DataFrame do Spark.
def data_aggregation(df):
    try:
        # Agrupando por tipo de cervejaria e localização
        aggregated_df = df.groupby(['brewery_type', 'city', 'state_province', 'country']).size().reset_index(name='brewery_count')
        return aggregated_df
    except Exception as e:
        logging.error(f"Erro ao realizar a agregação de dados: {e}")
        raise

In [ ]:
def data_aggregation_spark(df):
    try:
        # Mesmo agrupamento da versão pandas: groupby(...).size() -> groupBy(...).count()
        aggregated_df = (
            df.groupBy("brewery_type", "city", "state_province", "country")
              .agg(F.count("*").alias("brewery_count"))
        )
        return aggregated_df
    except Exception as e:
        logging.error(f"Erro ao realizar a agregação de dados: {e}")
        raise

Agora aplique a função na **Prata**, salve o resultado em **Delta** em `gold/breweries_por_local` e mostre as 10 linhas com maior `brewery_count`.

In [ ]:
gold_path = os.path.join(base_path, "gold", "breweries_por_local")

df_silver = spark.read.format("delta").load(silver_path)
df_gold = data_aggregation_spark(df_silver)

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)

df_gold_lida = spark.read.format("delta").load(gold_path)
print("Linhas na Ouro:", df_gold_lida.count())
# Conferência: a soma de brewery_count tem que bater com o total da Prata
print("Soma brewery_count:", df_gold_lida.agg(F.sum("brewery_count")).first()[0], "| Prata:", df_silver.count())

# Conferência cruzada com a função pandas original
pdf_ref = data_aggregation(df_silver.toPandas())
pdf_spark = df_gold_lida.toPandas()
chaves = ["brewery_type", "city", "state_province", "country"]
iguais = (pdf_ref.sort_values(chaves).reset_index(drop=True)
          .equals(pdf_spark[pdf_ref.columns].sort_values(chaves).reset_index(drop=True).astype(pdf_ref.dtypes.to_dict())))
print("Resultado igual ao pandas:", iguais)

df_gold_lida.orderBy(F.desc("brewery_count"), "country", "state_province", "city").show(10, truncate=False)

### ✏️ Q4. Para quem é a Ouro?
**(a)** Por que não deixar o dashboard ler a Prata e fazer essa agregação na hora? **(b)** Cite dois consumidores possíveis dessa tabela Ouro e o que cada um faria com ela.

**✏️ Sua resposta:**

**(a)** Porque a Ouro já entrega a resposta pronta e consistente. Se cada dashboard agregasse a Prata na hora: toda consulta reprocessaria a base inteira (lento e caro quando a base crescer), cada ferramenta poderia implementar a regra de um jeito diferente (ex.: uma conta nulos, outra não) e teríamos números divergentes, e o consumidor precisaria conhecer detalhes técnicos da Prata. A Ouro é a **fonte única da verdade** para essa pergunta de negócio, calculada uma vez e lida muitas vezes.

**(b)**
1. **Time comercial/expansão** de um fornecedor de insumos (malte, lúpulo): usaria o dashboard para ver quais cidades/estados têm mais microcervejarias e priorizar visitas de vendas.
2. **Time de dados/ML ou um assistente de IA**: usaria a tabela como feature ou contexto, por exemplo para responder "quantas brewpubs existem em Oregon?" ou para alimentar um modelo de potencial de mercado por região.

---
## 4. 🧊 Bônus Delta: viagem no tempo

O Delta guarda um **histórico de versões** no `_delta_log`. Rode a célula abaixo (ela já está pronta) e depois responda.

In [ ]:
from delta.tables import DeltaTable

# Reescreve a Prata uma segunda vez (gera uma nova versão)
spark.read.format("delta").load(silver_path).write.format("delta").mode("overwrite").save(silver_path)

DeltaTable.forPath(spark, silver_path).history().select("version", "timestamp", "operation").show(truncate=False)

# Ler a versão 0, como estava no início
spark.read.format("delta").option("versionAsOf", 0).load(silver_path).count()

### ✏️ Q5. Time travel
Em que situação real de trabalho você usaria `versionAsOf`? Dê um exemplo e explique **como isso é possível** em um storage onde os objetos não são editados no lugar.

**✏️ Sua resposta:**

Exemplo: um job da Prata roda com um bug (digamos, uma regra de limpeza que apagou o `brewery_type` de várias linhas) e o dashboard de segunda-feira mostra números estranhos. Com `versionAsOf` eu leio a versão anterior ao job ruim, comparo com a atual para achar exatamente o que mudou e, se precisar, restauro (`RESTORE TABLE ... TO VERSION AS OF n`). Também uso para **auditoria/reprodutibilidade**: "com quais dados o relatório do mês passado foi gerado?" ou para treinar de novo um modelo sobre a mesma foto dos dados.

Funciona porque o Delta **nunca edita arquivo no lugar**, o que combina com object storage (objetos são imutáveis). Cada escrita cria **novos** arquivos Parquet e um novo JSON no `_delta_log` dizendo "a versão N é composta pelos arquivos X, Y, Z". Os arquivos antigos continuam lá. Ler a versão 0 é só seguir o log até o commit 0 e ler os arquivos que ele listava. Isso vale até o `VACUUM` apagar os arquivos antigos (por padrão os com mais de 7 dias de retenção).

---
## 5. 🪣 Arquitetura: Object Storage

Estas perguntas não têm código. Elas medem se você entendeu **onde** e **como** os dados vivem.

### ✏️ Q6. Bucket, objeto e "pasta"
Explique a diferença entre **object storage** (S3, MinIO, GCS) e um **sistema de arquivos** comum. Por que se diz que em object storage "pasta não existe de verdade"? O que isso muda na hora de **renomear** ou **listar** dados?

**✏️ Sua resposta:**

Num **sistema de arquivos** existe uma árvore real: diretórios são entidades com metadados, você pode mover/renomear uma pasta e o SO altera só um ponteiro, e dá para editar um pedaço de um arquivo no lugar. No **object storage** o modelo é plano: um *bucket* guarda *objetos* identificados por uma *chave* (string) e você basicamente faz PUT, GET, DELETE e LIST do objeto inteiro, sem editar no meio.

"Pasta não existe" porque a `/` é só um caractere da chave. `bronze/bs_bronze_delta/part-0.parquet` é um nome; o console mostra como pasta porque agrupa as chaves por prefixo. Se apagar todos os objetos com aquele prefixo, a "pasta" some.

Consequências:
- **Renomear** uma "pasta" não é uma operação: é copiar cada objeto para a nova chave e apagar o antigo. Com milhares de arquivos isso é lento, caro e **não atômico** (pode falhar no meio). Por isso o Delta não depende de rename para fazer commit.
- **Listar** é uma chamada de API paginada por prefixo (ex.: até 1.000 chaves por requisição no S3). Listar diretórios com muitos arquivos é lento e custa requisições; por isso o Delta lê a lista de arquivos do `_delta_log` em vez de listar o bucket.

### ✏️ Q7. Storage separado do processamento
Spark e object storage são sistemas separados, ligados pela rede. **Cite uma vantagem e um custo** (ou risco) dessa separação. Se quiser, comente também o problema dos "arquivos pequenos".

**✏️ Sua resposta:**

**Vantagem:** escalamos e pagamos cada lado independentemente. O dado fica num storage barato, durável e praticamente infinito; o cluster Spark pode ser criado só na hora do job e desligado depois (ou vários clusters/ferramentas diferentes, como Spark, Trino, Athena, lendo os mesmos dados). Não preciso de um cluster ligado 24h só para "segurar" os dados.

**Custo/risco:** todo dado passa pela **rede**, então a leitura tem mais latência que um disco local, há custo por requisição (GET/LIST) e às vezes de transferência entre regiões, e a performance passa a depender da rede e das cotas da API.

**Arquivos pequenos:** como cada arquivo é uma requisição com latência fixa, ler 100 mil arquivos de 10 KB é muito mais lento (e caro) do que ler 100 arquivos de 128 MB com o mesmo volume total. Também pesa nos metadados (log do Delta, listagens) e no planejamento das tasks do Spark. A solução é compactar (`OPTIMIZE` no Delta, `coalesce/repartition` antes de gravar) e mirar arquivos na faixa de ~128 MB a 1 GB.

### ✏️ Q8. Do Colab para o S3
Nesta atividade gravamos em `/content/datalake`, no disco da máquina do Colab. **(a)** O que acontece com esses dados quando a sessão do Colab é encerrada? **(b)** Se o destino fosse um bucket (`s3a://meu-bucket/bronze/...`), o que mudaria no código e na configuração? **(c)** Que cuidados de **segurança** você teria?

**✏️ Sua resposta:**

**(a)** A VM do Colab é temporária: quando a sessão acaba (ou expira por inatividade), o disco é descartado e tudo em `/content/datalake` (Bronze, Prata, Ouro e o histórico Delta) **se perde**. Teríamos que rodar tudo de novo. Para persistir no Colab precisaria gravar no Drive ou num bucket.

**(b)** No código, basicamente só os caminhos: `base_path = "s3a://meu-bucket"`, e não usaria mais `os.makedirs`/`os.walk` (são do disco local; no S3 as "pastas" nem precisam ser criadas, e para listar usaria `boto3` ou as APIs do Hadoop FS). Na configuração do Spark: adicionar os pacotes `hadoop-aws` (e o SDK da AWS) compatíveis com a versão do Hadoop, configurar `spark.hadoop.fs.s3a.impl`, o provedor de credenciais e, no caso do MinIO, `fs.s3a.endpoint` e `fs.s3a.path.style.access=true`. Para o Delta em S3 com vários escritores simultâneos, configurar um LogStore com coordenação (ex.: DynamoDB), porque o S3 não garante "put-if-absent" do jeito que o Delta precisa no modo padrão.

**(c)** Segurança:
- **Nunca** colocar access key/secret no notebook ou no GitHub; usar IAM Role, variáveis de ambiente ou um gerenciador de segredos.
- **Menor privilégio**: a credencial do job só acessa o bucket/prefixo necessário (ex.: leitura na Bronze, escrita na Prata).
- Bucket **privado**, com bloqueio de acesso público ligado.
- **Criptografia** em repouso (SSE-S3/SSE-KMS) e em trânsito (HTTPS).
- Versionamento/políticas de retenção e logs de acesso (CloudTrail) para auditoria.
- Cuidado com dados pessoais (ex.: telefones) conforme a LGPD.

### ✏️ Q9. Projetando para escala
Imagine que a tabela Ouro passou a ter **500 milhões de linhas por ano**. Como você organizaria os arquivos no storage (formato, particionamento, tamanho dos arquivos)? Justifique pensando em **quem consulta** e **com que frequência**.

**✏️ Sua resposta:**

Pensando em quem consulta: a Ouro é lida por dashboards e analistas muitas vezes por dia, quase sempre filtrando por **país/região** e por **período**, enquanto é escrita poucas vezes (carga diária ou horária).

- **Formato:** continuar em **Delta** (Parquet colunar por baixo, comprimido). Leitura colunar faz o dashboard ler só as colunas que usa, e o Delta dá ACID, `MERGE` para cargas incrementais e estatísticas de min/max por arquivo para pular dados.
- **Particionamento:** por colunas de **baixa cardinalidade** que aparecem nos filtros: por exemplo uma data de referência (`ano/mes`, ou `dt` se a carga for diária) e `country`. **Não** particionaria por `city` (milhares de valores → milhões de arquivos pequenos). Para filtros por `state_province`/`city`/`brewery_type`, usaria **Z-ORDER** (ou *liquid clustering*), que agrupa dados parecidos dentro dos arquivos sem explodir o número de partições.
- **Tamanho dos arquivos:** mirar ~128 MB a 1 GB por arquivo. Rodar `OPTIMIZE` periodicamente para compactar os arquivos pequenos gerados pelas cargas incrementais e `VACUUM` para limpar versões antigas (respeitando a retenção do time travel).
- Para os dashboards mais usados, eu ainda criaria **agregados menores** (ex.: totais por país/mês), que respondem em milissegundos, deixando a tabela grande para consultas detalhadas.

---
## 6. 🤖 Uso de IA

### ✏️ IA-1. Você e o assistente de IA
Descreva **como você usou IA** nesta atividade (ou como usaria): para pesquisar os endereços, converter a função pandas para Spark, depurar erros? **(a)** Cole o prompt principal que você usou. **(b)** Como você **conferiu** que a resposta estava certa? **(c)** Teve algo que a IA errou ou inventou? Se não usou IA, explique como você usaria e quais riscos checaria (pense em endereços inventados e em código que "parece certo" mas dá resultado diferente).

**✏️ Sua resposta:**

Usei IA (Claude) como apoio em três pontos: entender a estrutura do notebook, converter a função pandas para Spark e localizar os endereços das 3 cervejarias.

**(a)** Prompt principal (resumido): *"Tenho este notebook de arquitetura medalhão com PySpark e Delta [conteúdo do notebook]. Me ajude a resolver as células de código da Prata e da Ouro e a encontrar o endereço real das cervejarias com address_1 nulo, indicando a fonte de cada endereço."*

**(b)** Como conferi:
- **Endereços:** não aceitei endereço sem fonte. Conferi cada um em páginas públicas (Brewbound para a 14 Lakes Brewery, site de turismo do condado de Mariposa para a 1850 Brewing) e se a cidade/CEP batiam com o registro.
- **Código:** rodei tudo e validei com números: a soma de `brewery_count` na Ouro tem que ser igual ao total de linhas da Prata (50); conferi que não sobrou endereço faltante e que não há ids duplicados; e comparei o resultado do Spark com o da função pandas original rodando `data_aggregation(df_silver.toPandas())`.

**(c)** Sim. Uma versão de referência da base trazia "12 W Main St" para a *12 West Brewing – Production Facility*, mas esse é o endereço do taproom no centro de Mesa (CEP 85201), não da unidade de produção do CEP 85207. A única fonte que achei (ABC15) diz apenas que a fábrica fica na Longbow Parkway perto da Recker Road, sem número. Por isso registrei como `"pesquisa manual (aproximado)"` em vez de inventar um número. É exatamente o tipo de erro que "parece certo" e passaria se eu não checasse. Outro ponto: a 1850 Brewing aparece em fontes com dois nomes para a mesma via (Yosemite All-Year Hwy / State Highway 140), e confirmei que são a mesma estrada.

### ✏️ IA-2. A qualidade do dado e a IA
Hoje as camadas Prata e Ouro costumam alimentar dashboards, modelos de ML e aplicações de IA generativa (como assistentes que respondem sobre os dados). Usando **exemplos desta base** (endereços faltantes, duplicados, tipos de cervejaria), explique como a **qualidade do dado** nas camadas anteriores afeta a **confiabilidade** do que a IA responde. O que você colocaria no pipeline para evitar respostas erradas?

**✏️ Sua resposta:**

Uma IA que responde sobre os dados não "sabe" mais que a tabela: ela repete o que a Prata/Ouro diz, só que com tom confiante. Exemplos nesta base:

- **Endereços faltantes:** se um assistente for perguntado "qual o endereço da 14 Lakes Brewery?" e o campo estiver nulo, um LLM tende a **inventar** um endereço plausível. E se eu preencher com um valor aproximado sem sinalizar, a IA vai apresentá-lo como exato. Por isso a coluna `address_fonte` importa: ela pode ser passada no contexto para a IA dizer "endereço aproximado".
- **Duplicados:** se a mesma cervejaria entrar duas vezes (mesmo `id` ou mesmo nome/endereço com ids diferentes), a Ouro conta em dobro e a IA responde "existem 2 micro em Mesa" quando há 1. Um modelo de ML também aprenderia uma distribuição errada.
- **Tipos de cervejaria:** a pergunta de negócio depende de `brewery_type`. Valores nulos, com grafia diferente ("Micro" vs "micro") ou categorias que mudam de significado (ex.: `closed`, `planning`, `proprietor`) geram grupos separados ou contagens de cervejarias que nem estão funcionando, e a IA responde sobre "cervejarias existentes" incluindo as fechadas.
- **Cobertura:** esta amostra tem 49 cervejarias dos EUA e 1 da Irlanda. Uma IA perguntada "qual país tem mais cervejarias?" vai responder com base só nessa amostra. Sem metadados dizendo que é uma amostra, a resposta parece um fato global.

O que eu colocaria no pipeline:
1. **Testes de qualidade automáticos** entre camadas (unicidade de `id`, não nulos nas chaves, valores de `brewery_type` dentro de um domínio permitido, checagem de que a soma da Ouro bate com a Prata), **bloqueando** a publicação se falharem.
2. **Quarentena** dos registros inválidos em vez de descartar ou inventar.
3. **Linhagem e flags** (`address_fonte`, data de carga, versão Delta) expostas para a IA, com instrução para responder "não sei/dado aproximado" quando o campo for nulo ou não original.
4. **Documentação/catálogo** (descrição das colunas, significado dos tipos, cobertura da base) usada como contexto para o assistente.
5. **Avaliação** periódica das respostas da IA contra perguntas com resposta conhecida calculada direto da Ouro.

---
## 7. 🏁 Bônus opcional: gráfico

Crie um gráfico (Spark + matplotlib, ou pandas) com os dados da Ouro. Sugestões: top 10 países por número de cervejarias, ou distribuição por `brewery_type`.

> Dica: use `.toPandas()` **só depois de agregar**, para trazer poucas linhas ao driver.

In [ ]:
import matplotlib.pyplot as plt

df_gold = spark.read.format("delta").load(gold_path)

# Agrega no Spark e só depois traz para o pandas (poucas linhas)
por_tipo = (df_gold.groupBy("brewery_type")
            .agg(F.sum("brewery_count").alias("total"))
            .orderBy(F.desc("total"))
            .toPandas())

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(por_tipo["brewery_type"], por_tipo["total"])
ax.invert_yaxis()
ax.set_xlabel("Número de cervejarias")
ax.set_title("Cervejarias por tipo (camada Ouro)")
for i, v in enumerate(por_tipo["total"]):
    ax.text(v + 0.2, i, str(v), va="center")
plt.tight_layout()
plt.show()

---
## ✅ Autoverificação

Rode esta célula antes de entregar. Ela confere **se os arquivos existem** e mostra as contagens. Ela não diz se o seu resultado está certo.

In [ ]:
checagens = {
    "Bronze": os.path.join(base_path, "bronze", "bs_bronze_delta"),
    "Prata ": os.path.join(base_path, "silver", "breweries_silver"),
    "Ouro  ": os.path.join(base_path, "gold",   "breweries_por_local"),
}
for nome, caminho in checagens.items():
    try:
        n = spark.read.format("delta").load(caminho).count()
        print(f"✅ {nome}: {n:,} linhas  ({os.path.relpath(caminho, base_path)})")
    except Exception as e:
        print(f"❌ {nome}: não encontrada em {os.path.relpath(caminho, base_path)}")

print("\nLembrete: responda TODAS as perguntas ✏️ (Q1 a Q9, IA-1 e IA-2) antes de enviar.")